# Run Airflow Operator

This notebook checks if a Google Cloud Run Job and its corresponding GCS bucket exist, creates them if necessary, triggers execution of the job with environment variables overrides to run an Airflow task, and monitors status until completion.

In [ ]:
# Parameter Cell
PROJECT_ID = "__PLACEHOLDER_PROJECT_ID__"
LOCATION = "us-central1"
JOB_NAME = "airflow-operator-job"
SERVICE_ACCOUNT = "dataform-compiler@__PLACEHOLDER_PROJECT_ID__.iam.gserviceaccount.com"

In [ ]:
import time
import json
import urllib.request
import urllib.error
import google.auth
import google.auth.transport.requests

# --- Helper Functions ---

def get_access_token():
    """Obtains an active Google Cloud API access token."""
    try:
        credentials, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/cloud-platform']
        )
        auth_req = google.auth.transport.requests.Request()
        credentials.refresh(auth_req)
        return credentials.token
    except Exception as e:
        print(f"Failed to obtain access token: {e}")
        raise e

def make_request(url, headers, method="GET", body=None):
    """Performs an HTTP request using urllib and returns the JSON response."""
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req) as response:
            res_body = response.read().decode("utf-8")
            return json.loads(res_body) if res_body else {}
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode('utf-8')
        raise urllib.error.HTTPError(e.url, e.code, f"{e.msg}: {err_msg}", e.hdrs, e.fp)

def get_project_number(project_id, headers):
    """Retrieves numeric project number from metadata server or REST API."""
    # 1. Try metadata server
    try:
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/numeric-project-id",
            headers={"Metadata-Flavor": "Google"},
            timeout=2
        )
        with urllib.request.urlopen(req) as response:
            return response.read().decode("utf-8").strip()
    except Exception:
        pass
        
    # 2. Try Cloud Resource Manager API
    try:
        url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{project_id}"
        res = make_request(url, headers, method="GET")
        return res.get("projectNumber")
    except Exception as e:
        print(f"Could not retrieve project number via API: {e}")
        
    # 3. Fallback default
    if project_id == "__PLACEHOLDER_PROJECT_ID__":
        return "1031557682594"
    return None

def ensure_bucket_exists(project_id, bucket_name, location, headers):
    """Checks if GCS bucket exists, and creates it if missing."""
    bucket_url = f"https://storage.googleapis.com/storage/v1/b/{bucket_name}"
    try:
        make_request(bucket_url, headers, method="GET")
        print(f"GCS Bucket {bucket_name} already exists.")
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"GCS Bucket {bucket_name} does not exist. Creating it...")
            create_url = f"https://storage.googleapis.com/storage/v1/b?project={project_id}"
            body = {"name": bucket_name, "location": location}
            make_request(create_url, headers, method="POST", body=body)
            print(f"GCS Bucket {bucket_name} created successfully.")
        else:
            raise e

def ensure_job_exists(project_id, location, job_name, project_number, bucket_name, service_account, headers):
    """Checks if Cloud Run job exists, and creates it if missing."""
    job_url = f"https://{location}-run.googleapis.com/v2/projects/{project_id}/locations/{location}/jobs/{job_name}"
    try:
        make_request(job_url, headers, method="GET")
        print(f"Cloud Run Job {job_name} already exists.")
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"Cloud Run Job {job_name} does not exist. Creating it...")
            create_url = f"https://{location}-run.googleapis.com/v2/projects/{project_id}/locations/{location}/jobs?jobId={job_name}"
            job_body = {
                "template": {
                    "template": {
                        "containers": [{
                            "name": "runner",
                            "image": f"us-central1-docker.pkg.dev/{project_id}/composer-lite-images/csr-lite-runner:latest",
                            "env": [
                                {"name": "SERVERLESS_PROJECT_NUMBER", "value": project_number},
                                {"name": "SERVERLESS_BUCKET_NAME", "value": bucket_name}
                            ],
                            "resources": {
                                "limits": {"cpu": "1000m", "memory": "2Gi"}
                            }
                        }],
                        "maxRetries": 3,
                        "timeout": "600s",
                        "serviceAccount": service_account,
                        "executionEnvironment": "EXECUTION_ENVIRONMENT_GEN2"
                    }
                }
            }
            op = make_request(create_url, headers, method="POST", body=job_body)
            poll_operation(location, op["name"], headers, "Job creation")
            print(f"Cloud Run Job {job_name} created successfully.")
        else:
            raise e

def poll_operation(location, operation_name, headers, task_name="Operation"):
    """Polls a Cloud Run Long Running Operation (LRO) until done."""
    poll_url = f"https://{location}-run.googleapis.com/v2/{operation_name}"
    print(f"Waiting for {task_name}...")
    while True:
        op = make_request(poll_url, headers, method="GET")
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(f"{task_name} failed: {op['error']}")
            return op
        time.sleep(2)

def trigger_job(project_id, location, job_name, project_number, bucket_name, headers):
    """Triggers the Cloud Run job with environment overrides and returns the execution name."""
    run_url = f"https://{location}-run.googleapis.com/v2/projects/{project_id}/locations/{location}/jobs/{job_name}:run"
    print(f"Triggering execution of job {job_name} with environment overrides...")
    
    body = {
        "overrides": {
            "containerOverrides": [
                {
                    "name": "runner",
                    "env": [
                        {
                            "name": "SERVERLESS_PROJECT_NUMBER",
                            "value": project_number
                        },
                        {
                            "name": "SERVERLESS_BUCKET_NAME",
                            "value": bucket_name
                        },
                        {
                            "name": "SERVERLESS_DAG_ID",
                            "value": "my-serverless-dag"
                        },
                        {
                            "name": "SERVERLESS_RUN_ID",
                            "value": "my-serverless-run"
                        },
                        {
                            "name": "SERVERLESS_TASK_ID",
                            "value": "my-serverless-task"
                        },
                        {
                            "name": "SERVERLESS_TRY_NUMBER",
                            "value": "1"
                        },
                        {
                            "name": "SERVERLESS_DAG_FILE",
                            "value": "__PLACEHOLDER_SERVERLESS_DAG_FILE_B64__"
                        }
                    ]
                }
            ]
        }
    }
    
    op = make_request(run_url, headers, method="POST", body=body)
    op = poll_operation(location, op["name"], headers, "Job trigger/startup")
    
    execution_response = op.get("response", {})
    execution_name = execution_response.get("name") or op.get("metadata", {}).get("name")
    if not execution_name:
        raise RuntimeError(f"Could not retrieve execution name from started job operation: {op}")
    return execution_name

def poll_execution(location, execution_name, headers):
    """Polls a Cloud Run Job execution until completed."""
    exec_url = f"https://{location}-run.googleapis.com/v2/{execution_name}"
    print(f"Monitoring execution {execution_name}...")
    while True:
        execution = make_request(exec_url, headers, method="GET")
        if execution.get("completionTime"):
            return execution
        print("Execution is still running... waiting 10 seconds")
        time.sleep(10)

# --- Main Flow ---

def main():
    global PROJECT_ID, SERVICE_ACCOUNT
    
    # 1. Setup Auth
    access_token = get_access_token()
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    # 2. Auto-detect Project ID if needed
    if not PROJECT_ID or PROJECT_ID == "__PLACEHOLDER_PROJECT_ID__":
        try:
            _, auto_project_id = google.auth.default()
            if auto_project_id:
                PROJECT_ID = auto_project_id
        except Exception as e:
            print(f"Could not auto-detect project ID: {e}. Using default: {PROJECT_ID}")

    # Resolve service account
    if not SERVICE_ACCOUNT:
        SERVICE_ACCOUNT = f"dataform-compiler@{PROJECT_ID}.iam.gserviceaccount.com"
    else:
        if "@__PLACEHOLDER_PROJECT_ID__.iam.gserviceaccount.com" in SERVICE_ACCOUNT:
            SERVICE_ACCOUNT = SERVICE_ACCOUNT.replace("__PLACEHOLDER_PROJECT_ID__", PROJECT_ID)

    print(f"Target Project: {PROJECT_ID}")
    print(f"Target Location: {LOCATION}")
    print(f"Target Job Name: {JOB_NAME}")
    print(f"Target Service Account: {SERVICE_ACCOUNT}")

    # 3. Retrieve numeric project number
    project_number = get_project_number(PROJECT_ID, headers)
    if not project_number:
        raise RuntimeError("Failed to retrieve project number for setting up job environment.")
    print(f"Resolved Project Number: {project_number}")

    # 4. Check/Create Bucket
    bucket_name = f"csr-lite-bucket-{project_number}"
    ensure_bucket_exists(PROJECT_ID, bucket_name, LOCATION, headers)

    # 5. Check/Create Cloud Run Job
    ensure_job_exists(PROJECT_ID, LOCATION, JOB_NAME, project_number, bucket_name, SERVICE_ACCOUNT, headers)

    # 6. Trigger Job Execution with overrides
    execution_name = trigger_job(PROJECT_ID, LOCATION, JOB_NAME, project_number, bucket_name, headers)
    print(f"Job execution started: {execution_name}")

    # 7. Monitor Job Execution
    execution = poll_execution(LOCATION, execution_name, headers)
    
    succeeded_count = int(execution.get("succeededCount", 0))
    failed_count = int(execution.get("failedCount", 0))
    cancelled_count = int(execution.get("cancelledCount", 0))

    print(f"Tasks: Succeeded: {succeeded_count}, Failed: {failed_count}, Cancelled: {cancelled_count}")
    if failed_count > 0:
        raise RuntimeError(f"Cloud Run job failed with {failed_count} failed tasks.")
    print("Cloud Run job completed successfully!")

# Run the flow
main()